In [5]:
import pandas as pd

loans_clean = pd.read_csv("../raw/loans_raw.csv")
print(loans_clean.shape)

(11236, 30)


In [10]:
loans_clean = loans_clean.copy()
dups = loans_clean.duplicated().sum()
print(f"Duplicate rows: {dups}")
dups_loans = loans_clean.duplicated(subset=["credit_number"]).sum()
print(f"Duplicate credit_number: {dups_loans}")

Duplicate rows: 0
Duplicate credit_number: 0


In [12]:
date_cols = [
    "agreement_signing_date",
    "board_approval_date",
    "closed_date_most_recent",
    "effective_date_most_recent",
    "end_of_period",
    "first_repayment_date",
    "last_disbursement_date",
    "last_repayment_date"
]

for col in date_cols:
    loans_clean[col] = pd.to_datetime(loans_clean[col], format="%d-%b-%Y", errors="coerce")

print(loans_clean[date_cols].dtypes)
print(f"\nFailed parses per column:")
print(loans_clean[date_cols].isna().sum())

agreement_signing_date        datetime64[ns]
board_approval_date           datetime64[ns]
closed_date_most_recent       datetime64[ns]
effective_date_most_recent    datetime64[ns]
end_of_period                 datetime64[ns]
first_repayment_date          datetime64[ns]
last_disbursement_date        datetime64[ns]
last_repayment_date           datetime64[ns]
dtype: object

Failed parses per column:
agreement_signing_date         148
board_approval_date              0
closed_date_most_recent          2
effective_date_most_recent     110
end_of_period                    0
first_repayment_date          2608
last_disbursement_date        2051
last_repayment_date           2608
dtype: int64


In [15]:
##print(loans_clean.columns.tolist())
loans_clean.rename(columns={
    "borrowers_obligation_us_":    "borrowers_obligation_usd",
    "cancelled_amount_us_":        "cancelled_amount_usd",
    "credits_held_us_":            "credits_held_usd",
    "disbursed_amount_us_":        "disbursed_amount_usd",
    "due_3rd_party_us_":           "due_3rd_party_usd",
    "due_to_ida_us_":              "due_to_ida_usd",
    "exchange_adjustment_us_":     "exchange_adjustment_usd",
    "original_principal_amount_us_": "original_principal_amount_usd",
    "repaid_3rd_party_us_":        "repaid_3rd_party_usd",
    "repaid_to_ida_us_":           "repaid_to_ida_usd",
    "sold_3rd_party_us_":          "sold_3rd_party_usd",
    "undisbursed_amount_us_":      "undisbursed_amount_usd"
}, inplace=True)

print(loans_clean.columns.tolist())

['agreement_signing_date', 'board_approval_date', 'borrower', 'borrowers_obligation_usd', 'cancelled_amount_usd', 'closed_date_most_recent', 'country', 'country_code', 'credit_number', 'credit_status', 'credits_held_usd', 'currency_of_commitment', 'disbursed_amount_usd', 'due_3rd_party_usd', 'due_to_ida_usd', 'effective_date_most_recent', 'end_of_period', 'exchange_adjustment_usd', 'first_repayment_date', 'last_disbursement_date', 'last_repayment_date', 'original_principal_amount_usd', 'project_id', 'project_name', 'region', 'repaid_3rd_party_usd', 'repaid_to_ida_usd', 'service_charge_rate', 'sold_3rd_party_usd', 'undisbursed_amount_usd']


In [16]:
nulls = loans_clean.isnull().sum()
print(nulls[nulls > 0])

agreement_signing_date         148
closed_date_most_recent          2
effective_date_most_recent     110
first_repayment_date          2608
last_disbursement_date        2051
last_repayment_date           2608
project_name                     4
service_charge_rate           2603
dtype: int64


In [17]:
## Why is there 2603 null rows
no_rate = loans_clean[loans_clean["service_charge_rate"].isna()]
print(f"Missing service_charge_rate: {len(no_rate)}")
print(f"\ncredit_status breakdown for those loans:")
print(no_rate["credit_status"].value_counts())

Missing service_charge_rate: 2603

credit_status breakdown for those loans:
credit_status
Fully Disbursed    1422
Disbursing          962
Effective           117
Signed               36
Fully Cancelled      34
Terminated           32
Name: count, dtype: int64


In [21]:
print(no_rate["currency_of_commitment"].value_counts())
print(f"\nService charge rate stats (where available):")
print(loans_clean["service_charge_rate"].describe())

currency_of_commitment
XDR    2539
USD      51
EUR      12
XAF       1
Name: count, dtype: int64

Service charge rate stats (where available):
count    11236.000000
mean         0.696486
std          0.670247
min          0.000000
25%          0.000000
50%          0.750000
75%          0.750000
max          5.388260
Name: service_charge_rate, dtype: float64


The XDR is the key - 2,539 out of 2,603 missing rates are in XDR (Special Drawing Rights), which is the IMF's reserve currency used specifically for IDA concessional credits and grants. These are the World Bank's softest loans to the poorest countries , often zero or near-zero interest, so no service charge rate is recorded.

In [20]:
loans_clean.to_csv("../cleaned/loans_clean.csv", index=False)
print("Saved: ../cleaned/loans_clean.csv")

Saved: ../cleaned/loans_clean.csv


The rich countries fund IDA → poor countries borrow it → money flows back to rich country suppliers ??????